# 26. SRA チャットボット・デモ
Notebook 25のマザーボードアーキテクチャを利用し、ipywidgetsによるチャットUIを構築します。
ユーザーの入力に基づいてリアルタイムでルーターが適切なシナプスを選択し、応答を生成します。


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import ipywidgets as widgets
from IPython.display import display, clear_output

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LLM_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"Using device: {device}")


In [ ]:
class LLMSynapse(nn.Module):
    """汎用LLMシナプス (Neural)"""
    def __init__(self, d_model):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Linear(d_model * 4, d_model)
        )
        # Qwen Instructモデルをロード
        from transformers import AutoModelForCausalLM, AutoTokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        self.model = AutoModelForCausalLM.from_pretrained(LLM_MODEL_ID).to(device)
        self.model.eval()
    
    def forward(self, x, text_input=None):
        if text_input is not None:
            # Qwenのチャット形式でユーザー入力を渡して応答を生成
            system_message = {
                "role": "system",
                "content": "You are having a normal one-on-one conversation. Reply only to the user's latest message in one short natural sentence. Do not introduce yourself. Do not explain what you are. Do not write code, tutorials, numbered lists, or examples unless explicitly asked. Match the user's language.",
            }
            if isinstance(text_input, list):
                messages = [system_message] + text_input[-8:]
            else:
                messages = [system_message, {"role": "user", "content": text_input}]
            prompt = self.tokenizer.apply_chat_template(
                messages,
                add_generation_prompt=True,
                tokenize=False,
            )
            encoded = self.tokenizer(prompt, return_tensors="pt").to(device)
            with torch.no_grad():
                outputs = self.model.generate(
                    input_ids=encoded["input_ids"],
                    attention_mask=encoded["attention_mask"],
                    max_new_tokens=40,
                    do_sample=False,
                    repetition_penalty=1.1,
                    eos_token_id=self.tokenizer.eos_token_id,
                    pad_token_id=self.tokenizer.eos_token_id,
                )
            generated = outputs[0, encoded["input_ids"].shape[-1]:]
            response = self.tokenizer.decode(generated, skip_special_tokens=True).strip()
            for marker in ["```", "\n- ", "\n1.", "\n2.", "\n3."]:
                response = response.split(marker, 1)[0].strip()
            response = response.split("\n\n", 1)[0].strip()
            return response
        return self.net(x)

In [ ]:
class VectorDBSynapse(nn.Module):
    """事実検索シナプス (Retrieval)"""
    def __init__(self, d_model):
        super().__init__()
        self.is_neural = False
        self.db = {}
        self.d_model = d_model
        
    def add_fact(self, key, value):
        self.db[key] = value
        
    def forward(self, x, text_input=None):
        if text_input is not None:
            for k, v in self.db.items():
                if k.lower() in text_input.lower():
                    return f"DB_RESULT: {v}"
            return "DB_MISS: 知識ベースにありません"
        return x

In [ ]:
class CalculatorSynapse(nn.Module):
    """ルールベース計算機シナプス"""
    def __init__(self):
        super().__init__()
        self.is_neural = False
    def forward(self, x, text_input=None):
        if text_input is not None:
            try:
                ans = eval(text_input)
                return f"CALC_RESULT: {ans}"
            except:
                return "CALC_ERROR: 有効な数式ではありません"
        return x

In [ ]:
class LastTokenRouter(nn.Module):
    def __init__(self, d_model, num_synapses):
        super().__init__()
        self.classifier = nn.Linear(d_model, num_synapses, bias=False)
        
    def forward(self, x):
        last_token_embeds = x[:, -1, :]
        logits = self.classifier(last_token_embeds)
        probs = F.softmax(logits, dim=-1)
        return probs, logits

class SRAMotherboard(nn.Module):
    def __init__(self, d_model, vocab_size):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.synapses = nn.ModuleDict()
        self.synapse_names = []
        self.router = None 
        
    def add_synapse(self, name, synapse_module):
        self.synapses[name] = synapse_module
        self.synapse_names.append(name)
        old_router = self.router
        self.router = LastTokenRouter(self.d_model, len(self.synapse_names)).to(device)
        if old_router is not None:
            with torch.no_grad():
                self.router.classifier.weight[:len(self.synapse_names)-1, :] = old_router.classifier.weight.data
                
    def _is_failed_output(self, output):
        return isinstance(output, str) and output.startswith(("CALC_ERROR", "DB_MISS"))

    def _extract_math_expression(self, text):
        import re
        # 簡易的にテキストから数式部分を抽出するロジック
        pattern = r'[\d\s\+\-\*\/\(\)\.]+'
        matches = re.findall(pattern, text)
        for match in matches:
            clean = match.strip()
            if any(op in clean for op in ['+', '-', '*', '/']) and len(clean) >= 3:
                return clean
        return None

    def route_and_forward(self, input_ids, text_inputs=None, llm_messages=None):
        x = self.embedding(input_ids)
        probs, _ = self.router(x)
        
        batch_size = x.size(0)
        final_outputs = []
        routed_synapses = []
        
        for i in range(batch_size):
            text_input = text_inputs[i] if text_inputs else None
            
            # 💡 【多段階インテリジェント協調ルーティング (JSON抽出 + Python構築版)】
            if text_input and any(k in text_input for k in ["買", "個", "本", "合計", "いくら", "円", "キャンペーン", "クーポン"]):
                print("[SRA Motherboard] Multi-stage cooperative routing triggered!")
                
                # Phase 1: LLMに曖昧表現の解釈と数量の抽出をJSON形式で要求 (Few-shotで安定化)
                json_prompt = (
                    "Analyze the user request and output a clean JSON object containing the quantities of items and discounts.\n"
                    "Recognize these mappings:\n"
                    "- '赤い丸い果物', 'りんご', 'apple' -> 'apple'\n"
                    "- '黄色くて細長い果物', 'バナナ', 'banana' -> 'banana'\n"
                    "- 'オレンジ', 'orange' -> 'orange'\n"
                    "- '雨', '土砂降り', 'rain' -> 'rain' (boolean)\n"
                    "- 'HAPPY50' -> 'happy50' (boolean)\n\n"
                    "Output format:\n"
                    "{\n"
                    "  \"apple\": 3,\n"
                    "  \"banana\": 2,\n"
                    "  \"orange\": 0,\n"
                    "  \"rain\": true,\n"
                    "  \"happy50\": true\n"
                    "}\n"
                    "Do NOT output any other text or explanation. Output ONLY JSON.\n\n"
                    f"User Request: {text_input}"
                )
                
                # LLMでJSON文字列を抽出
                raw_json = self.synapses["GeneralLLM"](x[i:i+1], text_input=json_prompt)
                print(f"[SRA Motherboard] LLM Raw JSON Output: {raw_json}")
                
                try:
                    import json
                    import re
                    # Qwenが余計な解説を付けた場合のための防衛策：最初と最後の { } を切り出す
                    json_match = re.search(r'\{.*?\}', raw_json, re.DOTALL)
                    if json_match:
                        parsed_data = json.loads(json_match.group(0))
                    else:
                        raise ValueError("No JSON block found")
                        
                    # Phase 2: マザーボード（Python）側での安全な数式自動構築
                    apple_qty = int(parsed_data.get("apple", 0))
                    banana_qty = int(parsed_data.get("banana", 0))
                    orange_qty = int(parsed_data.get("orange", 0))
                    is_rainy = bool(parsed_data.get("rain", False))
                    is_happy50 = bool(parsed_data.get("happy50", False))
                    
                    # 数式文字列の作成 (電卓デモ表示用。単価はDB情報:リンゴ=150,バナナ=200,オレンジ=120)
                    base_expr = f"({150} * {apple_qty} + {200} * {banana_qty} + {120} * {orange_qty})"
                    formula = base_expr
                    if is_rainy:
                        formula = f"({formula}) * 0.9"
                    if is_happy50:
                        formula = f"({formula}) - 50"
                        
                    print(f"[SRA Motherboard] Safely constructed formula: {formula}")
                    
                    # Phase 3: Calculatorで正確な金額を計算 (絶対に構文エラーが起きない)
                    calc_result = None
                    if "Calculator" in self.synapses:
                        calc_out = self.synapses["Calculator"](x[i:i+1], text_input=formula)
                        if not self._is_failed_output(calc_out):
                            calc_result = calc_out.replace("CALC_RESULT: ", "").strip()
                            
                    # Phase 4: 電卓の計算結果を基に、最終的なおもてなし応答をLLMに作成させる
                    if calc_result and "GeneralLLM" in self.synapses:
                        # 適用された割引の内訳を作成
                        discounts_list = []
                        if is_rainy:
                            discounts_list.append("雨の日割引10%オフ")
                        if is_happy50:
                            discounts_list.append("クーポンHAPPY50で50円引き")
                        discounts_str = "、".join(discounts_list) if discounts_list else "なし"
                        
                        final_prompt = (
                            f"ユーザーの注文: '{text_input}'\n"
                            f"システムが正確に計算した合計金額: '{calc_result}円'\n"
                            f"適用された割引: {discounts_str}\n"
                            f"この結果に基づいて、どの曖昧な商品がどれに該当したか（例: 赤い丸い果物=リンゴ（単価150円）、黄色くて細長い果物=バナナ（単価200円）など）"
                            f"とその数量、および最終合計金額を、ユーザーに対して親切でおもてなしのある自然な日本語で答えてください。"
                        )
                        
                        synapse_input = final_prompt
                        if llm_messages is not None and len(llm_messages) > i:
                            adjusted_messages = list(llm_messages[i])
                            if adjusted_messages and adjusted_messages[-1]["role"] == "user":
                                adjusted_messages[-1] = {
                                    "role": "user",
                                    "content": final_prompt
                                }
                            synapse_input = adjusted_messages
                            
                        out = self.synapses["GeneralLLM"](x[i:i+1], text_input=synapse_input)
                        final_outputs.append(f"[Cooperative(LLM->VectorDB->Calculator->LLM)] {out}")
                        routed_synapses.append("Cooperative(LLM+DB+Calc)")
                        continue
                except Exception as e:
                    print(f"[SRA Motherboard] Error parsing or constructing formula: {e}. Falling back to normal routing.")
                    # 失敗した場合は通常ルーティングへフォールバック

            # 通常の動的ルーティング・フォールバック (従来のロジック)
            math_expr = self._extract_math_expression(text_input) if text_input else None
            calc_result = None
            
            if math_expr and "Calculator" in self.synapses:
                calc_out = self.synapses["Calculator"](x[i:i+1], text_input=math_expr)
                if not self._is_failed_output(calc_out):
                    calc_result = calc_out.replace("CALC_RESULT: ", "").strip()
            
            if calc_result and "GeneralLLM" in self.synapses:
                cooperative_hint = (
                    f"(参考情報: システムの計算モジュールが算出した結果、数式 '{math_expr}' の正解は '{calc_result}' です。 "
                    f"この値を用いて、ユーザーの指示に自然な日本語で答えてください。自分で再計算しないでください。)"
                )
                
                synapse_input = text_input
                if llm_messages is not None and len(llm_messages) > i:
                    adjusted_messages = list(llm_messages[i])
                    if adjusted_messages and adjusted_messages[-1]["role"] == "user":
                        user_original = adjusted_messages[-1]["content"]
                        adjusted_messages[-1] = {
                            "role": "user",
                            "content": f"{user_original}\n{cooperative_hint}"
                        }
                    synapse_input = adjusted_messages
                else:
                    synapse_input = f"{text_input}\n{cooperative_hint}"
                
                out = self.synapses["GeneralLLM"](x[i:i+1], text_input=synapse_input)
                final_outputs.append(f"[Cooperative(Calculator->GeneralLLM)] {out}")
                routed_synapses.append("Cooperative(Calc+LLM)")
                continue
                
            sample_probs = probs[i]
            sorted_indices = torch.argsort(sample_probs, descending=True)
            
            for rank, syn_idx in enumerate(sorted_indices):
                selected_synapse_name = self.synapse_names[syn_idx.item()]
                synapse = self.synapses[selected_synapse_name]
                synapse_input = text_input
                if selected_synapse_name == "GeneralLLM" and llm_messages is not None:
                    synapse_input = llm_messages[i]
                out = synapse(x[i:i+1], text_input=synapse_input)
                if self._is_failed_output(out):
                    continue

                prefix = f"[{selected_synapse_name}]"
                if rank > 0:
                    primary_name = self.synapse_names[sorted_indices[0].item()]
                    prefix = f"[{selected_synapse_name} fallback from {primary_name}]"
                final_outputs.append(f"{prefix} {out}")
                routed_synapses.append(selected_synapse_name)
                break
            else:
                primary_name = self.synapse_names[sorted_indices[0].item()]
                final_outputs.append(f"[{primary_name}] 全シナプスで処理に失敗しました")
                routed_synapses.append(primary_name)
                
        return final_outputs, routed_synapses[-1] if len(routed_synapses) == 1 else routed_synapses

In [ ]:
vocab_size = 1000
d_model = 128

model = SRAMotherboard(d_model=d_model, vocab_size=vocab_size).to(device)

# シナプスの登録
model.add_synapse("GeneralLLM", LLMSynapse(d_model).to(device))
model.add_synapse("Calculator", CalculatorSynapse())

vdb = VectorDBSynapse(d_model)
vdb.add_fact("Japan", "Tokyo")
vdb.add_fact("CEO", "John Doe")
vdb.add_fact("SynapticRouter", "動的に最適なモジュールへルーティングする次世代AIアーキテクチャです。")

# 💡 デモ用の協調データ（商品価格と割引ルール）
vdb.add_fact("リンゴ", "1個 150円")
vdb.add_fact("バナナ", "1房 200円")
vdb.add_fact("オレンジ", "1個 120円")
vdb.add_fact("雨の日キャンペーン", "雨の日は合計金額から10%オフ (0.9倍)")
vdb.add_fact("HAPPY50", "合計金額から50円引き")

model.add_synapse("VectorDB", vdb)

# 入力テキストをダミーIDに変換する関数
def text_to_ids(text, max_len=5):
    # デモ用の簡易的な変換
    ids = [(ord(c) * 31) % vocab_size for c in text]
    if len(ids) < max_len:
        ids += [0] * (max_len - len(ids))
    return torch.tensor([ids[:max_len]]).to(device)

# デモ用にルーターを学習させる
demo_data = [
    {"text": "15 * 4", "target": 1},
    {"text": "100 / 2", "target": 1},
    {"text": "2 + 2", "target": 1},
    {"text": "Japan", "target": 2},
    {"text": "CEO", "target": 2},
    {"text": "SynapticRouter", "target": 2},
    {"text": "Hello", "target": 0},
    {"text": "こんにちは", "target": 0},
    {"text": "How are you?", "target": 0},
]

optimizer = torch.optim.Adam(model.router.parameters(), lr=0.05)
criterion = nn.CrossEntropyLoss()

print("ルーターを学習中...")
for epoch in range(100):
    optimizer.zero_grad()
    loss = 0
    for d in demo_data:
        ids = text_to_ids(d["text"])
        x = model.embedding(ids)
        _, logits = model.router(x)
        loss += criterion(logits, torch.tensor([d["target"]]).to(device))
    loss.backward()
    optimizer.step()
print("学習完了！")

## チャットボット UI
以下のセルを実行すると、チャット画面が表示されます。
数式（例: `15 * 4`）や、知識（例: `Japan`, `SynapticRouter`）、または一般的な挨拶（例: `Hello`）を入力してみてください。


In [ ]:
# チャットUIの構築
import html

response_area = widgets.HTML(
    value='<div><strong>SRA:</strong> 入力してください</div>',
    layout=widgets.Layout(
        width='100%',
        min_height='96px',
        border='1px solid #d0d7de',
        padding='12px',
        margin='0 0 8px 0',
    ),
)
chat_output = widgets.Output()
chat_history = []

text_input = widgets.Text(
    value='',
    placeholder='メッセージを入力... (例: 15 * 4, Japan, Hello)',
    description='You:',
    disabled=False,
    layout=widgets.Layout(width='80%'),
)

send_button = widgets.Button(
    description='送信',
    disabled=False,
    button_style='info', 
    tooltip='Click to send',
    icon='paper-plane'
)

# 💡 クイック選択プロンプト例ボタンUI
example_label = widgets.HTML("<b>プロンプト例（クリックで即送信）:</b>")
btn_layout = widgets.Layout(margin='2px', width='auto')

btn_ex1 = widgets.Button(description="😊 通常対話: こんにちは！自己紹介をして", layout=btn_layout, button_style='primary')
btn_ex2 = widgets.Button(description="🔢 正確な計算: 15 * 4 + 100", layout=btn_layout, button_style='success')
btn_ex3 = widgets.Button(description="🛒 多段階協調: 赤い丸い果物3個と細長い黄色い果物2本、雨、HAPPY50", layout=btn_layout, button_style='warning')

def on_ex_clicked(b):
    if b == btn_ex1:
        text_input.value = "こんにちは！自己紹介をして。"
    elif b == btn_ex2:
        text_input.value = "次の計算をして。15 * 4 + 100"
    elif b == btn_ex3:
        text_input.value = "赤い丸い果物を3個と、黄色くて細長い果物を2本買います。今日あいにく土砂降りなんだけど何かキャンペーンある？ あとシークレットクーポン『HAPPY50』も使いたいです。"
    on_send_clicked(None)

btn_ex1.on_click(on_ex_clicked)
btn_ex2.on_click(on_ex_clicked)
btn_ex3.on_click(on_ex_clicked)

examples_box = widgets.VBox([
    example_label,
    widgets.HBox([btn_ex1, btn_ex2]),
    widgets.HBox([btn_ex3])
])

def _format_response(text, routed_synapse=None):
    safe_text = html.escape(text).replace('\n', '<br>')
    route_label = f" <span style='color:#57606a'>(via {html.escape(routed_synapse)})</span>" if routed_synapse else ''
    return f"<div><strong>SRA:</strong>{route_label}<br>{safe_text}</div>"

def on_send_clicked(b):
    user_text = text_input.value.strip()
    if not user_text:
        return
        
    # コマンド処理
    if user_text.startswith('/'):
        cmd_parts = user_text.split(None, 1)
        cmd = cmd_parts[0].lower()
        args = cmd_parts[1].strip() if len(cmd_parts) > 1 else ""
        
        response_text = ""
        routed_synapse = "SystemCommand"
        
        if cmd == "/adddb":
            if "=" in args:
                k, v = args.split("=", 1)
                k = k.strip()
                v = v.strip()
                if k and v:
                    if "VectorDB" in model.synapses:
                        model.synapses["VectorDB"].add_fact(k, v)
                        response_text = f"ベクトルDBにファクトを追加しました: {k} -> {v}"
                    else:
                        response_text = "エラー: VectorDBシナプスが見つかりません。"
                else:
                    response_text = "エラー: キーと値を空にすることはできません。"
            else:
                response_text = "使用方法: /adddb key=value (例: /adddb US=Washington)"
                
        elif cmd == "/listdb":
            if "VectorDB" in model.synapses:
                db_content = model.synapses["VectorDB"].db
                if db_content:
                    lines = [f"- {k}: {v}" for k, v in db_content.items()]
                    response_text = "ベクトルDBの中身:\n" + "\n".join(lines)
                else:
                    response_text = "ベクトルDBは空です。"
            else:
                response_text = "エラー: VectorDBシナプスが見つかりません。"
                
        elif cmd == "/deldb":
            k = args.strip()
            if k:
                if "VectorDB" in model.synapses:
                    vdb_module = model.synapses["VectorDB"]
                    if k in vdb_module.db:
                        del vdb_module.db[k]
                        response_text = f"ベクトルDBからファクトを削除しました: {k}"
                    else:
                        response_text = f"エラー: キー '{k}' がベクトルDBに見つかりません。"
                else:
                    response_text = "エラー: VectorDBシナプスが見つかりません。"
            else:
                response_text = "使用方法: /deldb key (例: /deldb US)"
                
        elif cmd == "/synapses":
            response_text = "登録されているシナプス一覧:\n"
            for idx, name in enumerate(model.synapse_names):
                syn = model.synapses[name]
                is_neural = getattr(syn, "is_neural", True)
                syn_type = "Neural" if is_neural else "Rule-based/Retrieval"
                response_text += f"{idx + 1}. {name} ({syn_type})\n"
                
        elif cmd == "/clear":
            chat_history.clear()
            chat_output.clear_output()
            response_text = "チャット履歴をクリアしました。"
            
        elif cmd == "/help":
            response_text = (
                "利用可能なコマンド一覧:\n"
                "- /adddb key=value: ベクトルDBに新しいファクトを追加します\n"
                "- /listdb: ベクトルDBに登録されているすべてのファクトを表示します\n"
                "- /deldb key: ベクトルDBから指定したファクトを削除します\n"
                "- /synapses: 現在登録されているすべてのシナプスを表示します\n"
                "- /clear: チャット履歴をクリアします\n"
                "- /help: このヘルプメッセージを表示します"
            )
        else:
            response_text = f"未知のコマンドです: {cmd}\n/help を入力すると利用可能なコマンドを表示します。"
            
        with chat_output:
            print(f"👤 あなた: {user_text}")
        response_area.value = _format_response(response_text, routed_synapse)
        text_input.value = ''
        return
        
    response_area.value = _format_response('考えています...')
        
    with chat_output:
        print(f"👤 あなた: {user_text}")
        
        # 入力をIDに変換してルーティング・推論
        ids = text_to_ids(user_text)
        llm_messages = chat_history + [{"role": "user", "content": user_text}]
        outputs, routed_synapse = model.route_and_forward(
            ids,
            text_inputs=[user_text],
            llm_messages=[llm_messages],
        )

        # 応答の表示
        assistant_reply = outputs[0].split('] ', 1)[1]
        response_area.value = _format_response(assistant_reply, routed_synapse)
        if routed_synapse in ["GeneralLLM", "Cooperative(Calc+LLM)", "Cooperative(LLM+DB+Calc)"]:
            chat_history.append({"role": "user", "content": user_text})
            chat_history.append({"role": "assistant", "content": assistant_reply})
            del chat_history[:-8]
        
    text_input.value = ''

send_button.on_click(on_send_clicked)
text_input.continuous_update = False
def _on_value_change(change):
    if change.get('name') == 'value' and change.get('type') == 'change':
        on_send_clicked(None)
text_input.observe(_on_value_change, names='value')

display(widgets.VBox([response_area, examples_box, widgets.HBox([text_input, send_button]), chat_output]))